# Stable Diffusion 1.5 Ultimate Workstation
This notebook has been upgraded to a full-featured AI Art Station:
- **Modes**: Text-to-Image & Image-to-Image.
- **Universal Loader**: CivitAI & Hugging Face support.
- **LoRA Support**: Mix and match styles with adjustable scale.
- **AI Upscaling**: 2x Upscale (EDSR) for high-resolution details.
- **Prompt Manager**: Save and recall your favorite prompt combinations.
- **History Browser**: View and delete generated images directly in the UI.
- **Drive Integration**: Seamlessly save to Google Drive.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors gradio omegaconf invisible-watermark opencv-contrib-python-headless huggingface_hub

In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler
import os
import requests
from datetime import datetime
from google.colab import drive
from PIL import Image
import json
import random
import shutil
import cv2
import numpy as np
from huggingface_hub import login

# --- Global State ---
pipe_t2i = None
pipe_i2i = None
current_model_path = ""
drive_mounted = False
HISTORY_DIR = "/content/sd_history"
DRIVE_ROOT = "/content/drive/MyDrive"
UPSCALER_MODEL_PATH = "/content/EDSR_x2.pb"
upscaler = None
current_lora_path = None

if not os.path.exists(HISTORY_DIR):
    os.makedirs(HISTORY_DIR)

# --- Helper Classes ---
class PromptManager:
    def __init__(self):
        # Determine path dynamically based on Drive availability
        self.local_path = "/content/saved_prompts.json"
        self.drive_path = os.path.join(DRIVE_ROOT, "SD_Outputs", "saved_prompts.json")

    def get_filepath(self):
        # Prefer Drive if mounted
        if os.path.exists(DRIVE_ROOT):
             # Ensure directory exists
            drive_dir = os.path.dirname(self.drive_path)
            if not os.path.exists(drive_dir):
                try: os.makedirs(drive_dir)
                except: pass
            
            # Sync logic: If local exists but Drive doesn't, copy to Drive
            if os.path.exists(self.local_path) and not os.path.exists(self.drive_path):
                try: shutil.copy(self.local_path, self.drive_path)
                except: pass
            
            return self.drive_path
        return self.local_path

    def load(self):
        filepath = self.get_filepath()
        if not os.path.exists(filepath):
            return {}
        try:
            with open(filepath, 'r') as f:
                return json.load(f)
        except:
            return {}

    def save(self, name, prompt, neg_prompt):
        data = self.load()
        data[name] = {"prompt": prompt, "neg_prompt": neg_prompt}
        
        filepath = self.get_filepath()
        with open(filepath, 'w') as f:
            json.dump(data, f)
        
        # Return a Gradio update object to refresh the dropdown choices
        return gr.update(choices=list(data.keys()))

    def get_prompts_list(self):
        return list(self.load().keys())

    def get_prompt(self, name):
        data = self.load()
        if name in data:
            return data[name]["prompt"], data[name]["neg_prompt"]
        return "", ""

prompt_manager = PromptManager()

# --- Core Functions ---
def mount_google_drive():
    global drive_mounted
    if not drive_mounted:
        try:
            drive.mount('/content/drive')
            drive_mounted = True
            return True
        except Exception as e:
            print(f"Error mounting drive: {e}")
            return False
    return True

def download_file(url, filename):
    print(f"Downloading {filename}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    # Try content-disposition for filename if not provided or generic
    final_name = filename
    if "content-disposition" in response.headers and (filename == "custom_model.safetensors" or filename == "custom_lora.safetensors"):
        import re
        fname = re.findall("filename=(.+)", response.headers["content-disposition"])
        if fname:
            extracted_name = fname[0].strip('"')
            # Sanitize filename to prevent path traversal
            final_name = os.path.basename(extracted_name)

    with open(final_name, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded to {final_name}")
    return final_name

def setup_upscaler():
    global upscaler
    if upscaler is not None:
        return
    
    if not os.path.exists(UPSCALER_MODEL_PATH):
        # Download EDSR_x2.pb (approx 38MB)
        url = "https://github.com/Saafke/EDSR_Tensorflow/raw/master/models/EDSR_x2.pb"
        download_file(url, UPSCALER_MODEL_PATH)
    
    try:
        # Use direct submodule access to be safe
        upscaler = cv2.dnn_superres.DnnSuperResImpl_create()
        upscaler.readModel(UPSCALER_MODEL_PATH)
        upscaler.setModel("edsr", 2)
    except Exception as e:
        print(f"Error setting up upscaler: {e}")

def upscale_image_cv2(image_pil):
    setup_upscaler()
    if upscaler is None:
        print("Upscaler not available.")
        return image_pil
    
    # Convert PIL to CV2 (BGR)
    img = np.array(image_pil)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    # Upscale
    result = upscaler.upsample(img)
    
    # Convert back to PIL (RGB)
    result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    return Image.fromarray(result)

def load_model(model_path_or_url, hf_token):
    global pipe_t2i, pipe_i2i, current_model_path
    
    # Authenticate if token provided
    if hf_token.strip():
        try:
            login(token=hf_token)
        except Exception as e:
            print(f"Warning: Auth failed: {e}")

    if pipe_t2i is not None and model_path_or_url == current_model_path:
        return "Model already loaded."

    print(f"Loading model: {model_path_or_url}...")
    
    try:
        final_path = model_path_or_url
        is_repo = not model_path_or_url.startswith("http") and "/" in model_path_or_url
        
        if not is_repo:
            if model_path_or_url.startswith("http"):
                final_path = download_file(model_path_or_url, "custom_model.safetensors")

        # Load Pipeline (Text2Img)
        if is_repo:
            pipe_t2i = StableDiffusionPipeline.from_pretrained(
                final_path, 
                torch_dtype=torch.float16, 
                use_safetensors=True,
                use_auth_token=hf_token if hf_token else True # Use cached if empty, or explicit
            )
        else:
            pipe_t2i = StableDiffusionPipeline.from_single_file(
                final_path, torch_dtype=torch.float16
            )

        # Configure Scheduler
        pipe_t2i.scheduler = DPMSolverMultistepScheduler.from_config(
            pipe_t2i.scheduler.config, 
            use_karras_sigmas=True, 
            algorithm_type="dpmsolver++"
        )
        
        # Optimizations
        pipe_t2i.enable_model_cpu_offload()
        pipe_t2i.safety_checker = None
        
        # Create Img2Img Pipeline sharing components
        pipe_i2i = StableDiffusionImg2ImgPipeline(vae=pipe_t2i.vae, text_encoder=pipe_t2i.text_encoder, tokenizer=pipe_t2i.tokenizer, unet=pipe_t2i.unet, scheduler=pipe_t2i.scheduler, safety_checker=None, feature_extractor=None)
        
        current_model_path = model_path_or_url
        return "Model loaded successfully!"
        
    except Exception as e:
        return f"Error loading model: {str(e)}"

def load_lora(lora_path_or_url):
    global pipe_t2i, pipe_i2i, current_lora_path
    if pipe_t2i is None:
        return "Load a base model first."
        
    try:
        print(f"Loading LoRA: {lora_path_or_url}")
        final_path = lora_path_or_url
        
        if lora_path_or_url.startswith("http"):
            final_path = download_file(lora_path_or_url, "custom_lora.safetensors")
            
        # Unload previous adapters to prevent accumulation/conflicts (Simple 1-LoRA approach)
        try:
            pipe_t2i.unload_lora_weights()
        except: pass
        
        pipe_t2i.load_lora_weights(final_path)
        # Sync I2I pipe? Usually they share unet, so yes.
        # But just in case, adapters attach to unet/text_encoder which are shared objects.
        
        current_lora_path = lora_path_or_url
        return f"LoRA loaded: {os.path.basename(final_path)}"
    except Exception as e:
        return f"Error loading LoRA: {str(e)}"

def generate(mode, prompt, neg_prompt, img_input, steps, cfg, width, height, seed, randomize, num_images, save_drive, drive_folder, denoise, do_upscale, lora_scale):
    global pipe_t2i, pipe_i2i
    if pipe_t2i is None:
        return [None] * int(num_images), seed

    # Handle Seed
    if randomize:
        seed = random.randint(0, 2147483647)
    
    # Configure Cross Attention Scale for LoRA (Applied dynamically)
    # Note: 'scale' is supported in cross_attention_kwargs in newer diffusers
    cross_att_kwargs = {"scale": lora_scale} if lora_scale != 1.0 else None
    
    print(f"Generating {num_images} images in mode: {mode}")
    
    generated_images = []
    
    # Sequential Generation Loop (Batch Size 1) for safety
    for i in range(int(num_images)):
        # Advance seed for subsequent images if randomize was requested (or even if not, to vary batch?)
        # Standard practice: if random, new random. If fixed, fixed + i.
        current_seed = seed + i if not randomize else random.randint(0, 2147483647)
        if i == 0 and not randomize: current_seed = seed # Keep first seed exact
        
        generator = torch.Generator(device="cpu").manual_seed(int(current_seed))

        if mode == "txt2img":
            out = pipe_t2i(
                prompt, negative_prompt=neg_prompt, num_inference_steps=steps, 
                guidance_scale=cfg, width=width, height=height, 
                num_images_per_prompt=1, generator=generator,
                cross_attention_kwargs=cross_att_kwargs
            ).images[0]
        else: # img2img
            if img_input is None:
                continue
            init_image = img_input.resize((width, height))
            out = pipe_i2i(
                prompt, image=init_image, negative_prompt=neg_prompt, 
                num_inference_steps=steps, guidance_scale=cfg, strength=denoise,
                num_images_per_prompt=1, generator=generator,
                cross_attention_kwargs=cross_att_kwargs
            ).images[0]
        
        # Upscale
        if do_upscale:
            print(f"Upscaling image {i+1}...")
            out = upscale_image_cv2(out)
            
        generated_images.append(out)

    # Save Logic
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. Save to History (Colab Local)
    for i, img in enumerate(generated_images):
        fname = f"sd_{timestamp}_{i}.png"
        img.save(os.path.join(HISTORY_DIR, fname))
    
    # 2. Save to Drive (Optional)
    if save_drive and mount_google_drive():
        drive_path = f"/content/drive/MyDrive/{drive_folder}"
        os.makedirs(drive_path, exist_ok=True)
        for i, img in enumerate(generated_images):
            img.save(f"{drive_path}/sd_{timestamp}_{i}.png")

    return generated_images, seed

def get_history_images():
    files = sorted(os.listdir(HISTORY_DIR), reverse=True)
    paths = [os.path.join(HISTORY_DIR, f) for f in files if f.endswith('.png')]
    return paths

def select_image(evt: gr.SelectData):
    return evt.value['image']['path']

def delete_image(path):
    if not path:
        return get_history_images(), "No image selected."
    try:   
        filename = os.path.basename(path)
        real_path = os.path.join(HISTORY_DIR, filename)
        if os.path.exists(real_path):
            os.remove(real_path)
            return get_history_images(), "Image deleted."
        else:
            return get_history_images(), f"Image not found at {real_path}"
    except Exception as e:
        return get_history_images(), f"Error: {str(e)}"

# --- UI Layout ---
with gr.Blocks(theme=gr.themes.Soft(), title="SD Ultimate Workstation") as demo:
    gr.Markdown("# Stable Diffusion 1.5 Ultimate Workstation")
    
    with gr.Row():
        # --- Left Column: Inputs & Settings ---
        with gr.Column(scale=1):
            # Model Section
            with gr.Group():
                gr.Markdown("### 1. Model & LoRA")
                with gr.Row():
                    model_input = gr.Textbox(label="Base Model (URL/ID)", value="runwayml/stable-diffusion-v1-5")
                    hf_token = gr.Textbox(label="HF Token (Optional)", type="password", placeholder="For gated models")
                load_btn = gr.Button("Load Model", variant="secondary", size="sm")
                
                with gr.Row():
                    lora_input = gr.Textbox(label="LoRA (URL/ID)", placeholder="Optional")
                    lora_scale = gr.Slider(label="LoRA Scale", minimum=0.0, maximum=1.0, value=0.75)
                load_lora_btn = gr.Button("Load LoRA", variant="secondary", size="sm")
                status_text = gr.Textbox(label="Status", interactive=False, max_lines=1)

            # Generation Section
            with gr.Group():
                gr.Markdown("### 2. Prompt & Generate")
                # Mode Tabs
                mode_state = gr.State(value="txt2img")
                with gr.Tabs() as mode_tabs:
                    with gr.Tab("Text-to-Image", id="t2i_tab"): pass
                    with gr.Tab("Image-to-Image", id="i2i_tab"):
                        img_input = gr.Image(label="Input Image", type="pil", height=200)
                        denoise = gr.Slider(label="Denoise Strength", minimum=0.0, maximum=1.0, value=0.75)

                prompt = gr.Textbox(label="Prompt", lines=3, placeholder="Describe your masterpiece...")
                neg_prompt = gr.Textbox(label="Negative Prompt", lines=2, value="low quality, bad anatomy, worst quality")
                
                with gr.Row():
                     gen_btn = gr.Button("GENERATE", variant="primary", size="lg", scale=2)
                     num_images = gr.Slider(label="Count", minimum=1, maximum=10, step=1, value=1, scale=1)

            # Advanced Settings
            with gr.Accordion("Advanced Settings", open=True):
                with gr.Row():
                    width = gr.Slider(label="Width", minimum=256, maximum=1024, step=64, value=512)
                    height = gr.Slider(label="Height", minimum=256, maximum=1024, step=64, value=512)
                with gr.Row():
                    steps = gr.Slider(label="Steps", minimum=10, maximum=100, step=1, value=25)
                    cfg = gr.Slider(label="CFG", minimum=1, maximum=20, step=0.5, value=7.5)
                
                with gr.Row():
                    random_seed = gr.Checkbox(label="Random Seed", value=True)
                    seed = gr.Number(label="Seed", value=42, precision=0)
                
                do_upscale = gr.Checkbox(label="AI Upscale (2x)", value=False, info="Uses EDSR x2 (High Quality)")
                
                with gr.Row():
                    save_drive = gr.Checkbox(label="Save to Drive", value=False)
                    drive_folder = gr.Textbox(label="Folder", value="SD_Outputs", show_label=False)

            # Prompt Manager
            with gr.Accordion("Prompt Manager", open=False):
                style_name = gr.Textbox(label="Name")
                save_style_btn = gr.Button("Save Current")
                saved_styles_drop = gr.Dropdown(label="Load Style", choices=prompt_manager.get_prompts_list())
                load_style_btn = gr.Button("Load Selected")

        # --- Right Column: Output & History ---
        with gr.Column(scale=1):
            gr.Markdown("### Results")
            gallery = gr.Gallery(label="Output", columns=2, height=600)
            
            gr.Markdown("### History")
            with gr.Row():
                refresh_hist_btn = gr.Button("Refresh")
                del_img_btn = gr.Button("Delete Selected", variant="stop")
            hist_gallery = gr.Gallery(label="Saved Images", columns=4, allow_preview=True, height=300)
            hist_msg = gr.Textbox(label="System Info", show_label=False)
            selected_img_path = gr.State(value="")

    # --- Wiring ---
    # Loaders
    load_btn.click(load_model, [model_input, hf_token], [status_text])
    load_lora_btn.click(load_lora, [lora_input], [status_text])

    # Mode Switch
    t2i_tab = mode_tabs.children[0]
    i2i_tab = mode_tabs.children[1]
    t2i_tab.select(fn=lambda: "txt2img", outputs=mode_state)
    i2i_tab.select(fn=lambda: "img2img", outputs=mode_state)

    # Generate
    gen_btn.click(
        fn=generate,
        inputs=[mode_state, prompt, neg_prompt, img_input, steps, cfg, width, height, seed, random_seed, num_images, save_drive, drive_folder, denoise, do_upscale, lora_scale],
        outputs=[gallery, seed]
    )

    # History
    refresh_hist_btn.click(get_history_images, outputs=[hist_gallery])
    hist_gallery.select(select_image, None, selected_img_path)
    del_img_btn.click(delete_image, inputs=[selected_img_path], outputs=[hist_gallery, hist_msg])

    # Prompt Manager
    save_style_btn.click(
        fn=lambda n, p, np: prompt_manager.save(n, p, np),
        inputs=[style_name, prompt, neg_prompt],
        outputs=[saved_styles_drop]
    )
    def load_style_wrapper(name):
        p, np = prompt_manager.get_prompt(name)
        return p, np
    load_style_btn.click(load_style_wrapper, [saved_styles_drop], [prompt, neg_prompt])

demo.launch(share=True, debug=True)